# 03. Modelo Base (Baseline Heurístico) - Isolation Forest

**Proyecto:** AIOps Multivariate Anomaly Detection  
**Objetivo:** Entrenar y evaluar un modelo de referencia clásico no supervisado (`IsolationForest`) para establecer la línea base de detección y contrastar su desempeño frente a redes neuronales profundas.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import joblib

print("Entorno Isolation Forest listo.")

Entorno Isolation Forest listo.


## 1. Carga de Datos Procesados

In [2]:
proc_dir = os.path.join('..', 'data', 'processed')
if os.path.exists(os.path.join(proc_dir, 'X_train.npy')):
    X_train_win = np.load(os.path.join(proc_dir, 'X_train.npy'))
    X_test_win = np.load(os.path.join(proc_dir, 'X_test.npy'))
else:
    np.random.seed(42)
    X_train_win = np.random.normal(0, 1, (1000, 60, 38))
    X_test_win = np.random.normal(0, 1, (500, 60, 38))

# Aplanar ventanas a 2280 dimensiones para modelos tabulares
N_train, W, C = X_train_win.shape
X_train_flat = X_train_win.reshape(N_train, -1)
X_test_flat = X_test_win.reshape(X_test_win.shape[0], -1)

print("Dimensiones para Baseline:", X_train_flat.shape)

Dimensiones para Baseline: (3941, 2280)


## 2. Ajuste de Isolation Forest

In [3]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.04,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_flat)

# Guardar baseline
joblib.dump(iso_forest, os.path.join('..', 'models', 'model_v1.pkl'))
print("Isolation Forest ajustado y serializado en models/model_v1.pkl")

Isolation Forest ajustado y serializado en models/model_v1.pkl


## 3. Inferencia y Puntuación de Anomalía

In [4]:
# Predicción (-1 anomalía, 1 normal)
preds_raw = iso_forest.predict(X_test_flat)
y_pred_baseline = np.where(preds_raw == -1, 1, 0)
anomaly_scores = -iso_forest.decision_function(X_test_flat)

print(f"Porcentaje de anomalías detectadas en test: {np.mean(y_pred_baseline)*100:.2f}%")
print(f"Rango de anomaly scores: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")

Porcentaje de anomalías detectadas en test: 4.09%
Rango de anomaly scores: [-0.0459, 0.0166]
